# Madhav E-Commerce Sales Analysis and Dashboard

## Imports
Libraries for data access, analysis, visualization, and dashboarding.

In [ ]:
# Data access + analysis stack
import kagglehub
from kagglehub import KaggleDatasetAdapter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Interactive visuals + dashboard components
import plotly.express as px
from dash import Dash, html, dcc, Input, Output

## Load Madhav Dataset (Kaggle)
Download the Orders and Details tables from the Kaggle dataset.

### Load `Orders` table
Order-level records with dates, customer info, and order attributes.

In [ ]:
# Load Orders table from the Kaggle dataset
file_path = "Orders.csv"

Orders_df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "amitkumar209/madhav-e-commerce-sales-dataset",
    file_path
)

# Quick look at the schema and first rows
Orders_df.head()

C:\Users\gaura\AppData\Local\Temp\ipykernel_24988\3352895207.py:3: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  Orders_df = kagglehub.load_dataset(


,Order ID,Order Date,CustomerName,State,City
0,B-26055,10-03-2018,Harivansh,Uttar Pradesh,Mathura
1,B-25993,03-02-2018,Madhav,Delhi,Delhi
2,B-25973,24-01-2018,Madan Mohan,Uttar Pradesh,Mathura
3,B-25923,27-12-2018,Gopal,Maharashtra,Mumbai
4,B-25757,21-08-2018,Vishakha,Madhya Pradesh,Indore


### Load `Details` table
Line-item records with product, category, sales, and profit fields.

In [ ]:
# Load line-item Details table
file_path = "Details.csv"

Details_df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "amitkumar209/madhav-e-commerce-sales-dataset",
    file_path,
)

# Quick look at the schema and first rows
Details_df.head()

C:\Users\gaura\AppData\Local\Temp\ipykernel_24988\1726229362.py:3: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  Details_df = kagglehub.load_dataset(


,Order ID,Amount,Profit,Quantity,Category,Sub-Category,PaymentMode
0,B-25681,1096,658,7,Electronics,Electronic Games,COD
1,B-26055,5729,64,14,Furniture,Chairs,EMI
2,B-25955,2927,146,8,Furniture,Bookcases,EMI
3,B-26093,2847,712,8,Electronics,Printers,Credit Card
4,B-25602,2617,1151,4,Electronics,Phones,Credit Card


## Merge Datasets
Join on `Order ID` and confirm the combined schema before analysis.

In [ ]:
# Join orders with item details on Order ID
df = pd.merge(Orders_df, Details_df, on="Order ID")
df.head()

,Order ID,Order Date,CustomerName,State,City,Amount,Profit,Quantity,Category,Sub-Category,PaymentMode
0,B-26055,10-03-2018,Harivansh,Uttar Pradesh,Mathura,5729,64,14,Furniture,Chairs,EMI
1,B-26055,10-03-2018,Harivansh,Uttar Pradesh,Mathura,671,114,9,Electronics,Phones,Credit Card
2,B-26055,10-03-2018,Harivansh,Uttar Pradesh,Mathura,443,11,1,Clothing,Saree,COD
3,B-26055,10-03-2018,Harivansh,Uttar Pradesh,Mathura,57,7,2,Clothing,Shirt,UPI
4,B-26055,10-03-2018,Harivansh,Uttar Pradesh,Mathura,227,48,5,Clothing,Stole,COD


In [ ]:
# Review column types and missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Order ID      1500 non-null   object
 1   Order Date    1500 non-null   object
 2   CustomerName  1500 non-null   object
 3   State         1500 non-null   object
 4   City          1500 non-null   object
 5   Amount        1500 non-null   int64 
 6   Profit        1500 non-null   int64 
 7   Quantity      1500 non-null   int64 
 8   Category      1500 non-null   object
 9   Sub-Category  1500 non-null   object
 10  PaymentMode   1500 non-null   object
dtypes: int64(3), object(8)
memory usage: 129.0+ KB


## Data Cleaning
Parse dates and create time-based features for trend analysis.

### Convert `Order Date` to datetime
Ensure dates are parsed correctly for sorting, grouping, and resampling.

In [ ]:
# Parse order date for time-based analysis
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True)
df["Order Date"].head()

0   2018-03-10
1   2018-03-10
2   2018-03-10
3   2018-03-10
4   2018-03-10
Name: Order Date, dtype: datetime64[ns]

#### Create an `Order Month` column
Store month names for seasonal and monthly trend charts.

In [ ]:
# Extract month name for trend charts
df["Order Month"] = pd.to_datetime(df["Order Date"]).dt.strftime("%b")
df["Order Month"].sample(5)

203    Mar
878    Jul
701    Apr
120    Oct
824    Feb
Name: Order Month, dtype: object

#### Create `Quater` (quarter) column
Derive quarter values for higher-level seasonal grouping.

In [ ]:
# Extract quarter for seasonal grouping
df["Quater"] = pd.to_datetime(df["Order Date"]).dt.quarter
df["Quater"].sample(5)

1485    1
738     1
384     3
575     4
1088    4
Name: Quater, dtype: int32